# 微带线综合入门（E6a HJ 闭式）

消费 `rfauto.core.synthesis`（Hammerstad-Jensen 正向 + brentq 反解，已合流闭式面）：给定层叠与目标阻抗求线宽，正向复核自洽。

- 全部数值出自确定性内核（铁律 7），零 EM 求解器、零真机依赖；
- 层叠为演示自建参数（非 configs/materials.yaml 档案值）。

In [ ]:
from rfauto.core.synthesis import Stackup, forward_z0, inverse_width

stack = Stackup(name="demo_er348_h0508", epsilon_r=3.48,
                thickness_mm=0.508, loss_tangent=0.0037)
w50, z0_actual, status = inverse_width(50.0, 10.0, stack)
print(f"10 GHz 50Ω 线宽: {w50:.4f} mm "
      f"(回代 Z0={z0_actual:.3f} Ω, status={status})")
z0_check, er_eff = forward_z0(w50, 10.0, stack)
print(f"正向复核: Z0={z0_check:.4f} Ω, εeff={er_eff:.4f}")
assert abs(z0_check - 50.0) < 0.5  # 引擎自洽断言口径

In [ ]:
# 阻抗-线宽响应性（Z0 单调降，引擎验收②同款守卫）
widths = [0.2, 0.5, 1.0, 2.0, 4.0]
print(f"{'W [mm]':>8} {'Z0 [Ω]':>10} {'εeff':>8}")
z0s = []
for w in widths:
    z0, er = forward_z0(w, 10.0, stack)
    z0s.append(z0)
    print(f"{w:>8.2f} {z0:>10.2f} {er:>8.4f}")
assert all(a > b for a, b in zip(z0s, z0s[1:])), "Z0 必须随 W 单调降"

In [ ]:
# 同层叠不同频率的 50Ω 线宽
for f_ghz in (1.0, 2.4, 5.8, 10.0, 28.0):
    w, z0, st = inverse_width(50.0, f_ghz, stack)
    print(f"f={f_ghz:>5.1f} GHz → W={w:.4f} mm ({st})")